# 🎬 SHORT MAKER — All-In-One Pipeline

Tạo video Short dạy tiếng Anh viral **hoàn toàn tự động**.

### Quy trình:
1. Chạy **Cell 1** (cài đặt, 1 lần duy nhất)
2. Nhập cấu hình ở **Cell 2**
3. Chạy **Cell 3** → Video tự động tạo xong
4. Chạy **Cell 4** → Tải video về máy

### Yêu cầu:
- **Colab**: Runtime → Change runtime type → **T4 GPU**
- **Máy local**: Chạy `python bridge_local.py` + mở TurboFlow extension trên Edge
- Hoặc bỏ qua ảnh AI → dùng ảnh Pexels stock (miễn phí)
- Upload file giọng mẫu `.wav` lên Colab (cột trái 📁)

In [ ]:
# @title ⚙️ CELL 1: CÀI ĐẶT (chạy 1 lần, ~3 phút)
import os, subprocess, sys

print('⏳ Đang cài đặt thư viện...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'qwen-tts', 'huggingface_hub', 'pydub', 'openai-whisper',
    'pysrt', 'requests', 'playwright'], check=True,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

os.system('apt-get install -y -qq ffmpeg sox libsox-fmt-all 2>/dev/null')
os.system('playwright install chromium 2>/dev/null')
os.system('playwright install-deps 2>/dev/null')

from IPython.display import clear_output
clear_output()

# Quick import test
import torch, soundfile, whisper, pysrt, requests, json, re, time, shutil, zipfile
from pathlib import Path
from qwen_tts import Qwen3TTSModel
from playwright.sync_api import sync_playwright

print('✅ TẤT CẢ THƯ VIỆN ĐÃ SẴN SÀNG!')
print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('   → Upload file giọng mẫu (.wav) lên cột trái 📁')
print('   → Chạy Cell 2 để cấu hình')

In [ ]:
# @title 📝 CELL 2: CẤU HÌNH
# @markdown ---
# @markdown ### Chủ đề video
topic = 'How to use the word GET in English' # @param {type:'string'}
# @markdown ---
# @markdown ### API Keys
groq_api_key = '' # @param {type:'string'}
# @markdown > Groq key miễn phí: https://console.groq.com/keys
pexels_api_key = '' # @param {type:'string'}
# @markdown > Pexels key miễn phí: https://www.pexels.com/api/
# @markdown ---
# @markdown ### Kết nối TurboFlow (tùy chọn)
bridge_url = '' # @param {type:'string'}
# @markdown > Paste URL tunnel từ `bridge_local.py`. Bỏ trống = dùng ảnh Pexels.
# @markdown ---
# @markdown ### 🎤 Giọng đọc (Voice Clone)
file_giong_mau = 'yo.wav' # @param {type:'string'}
# @markdown > Upload file giọng mẫu (.wav) lên Colab (cột trái 📁), nhập tên file.
loi_thoai_giong_mau = 'Wait, have you ever noticed that time feels faster as we get older? It is kind of scary, right? But actually, there is a hidden logic behind it.' # @param {type:'string'}
# @markdown > Nội dung mà người trong file mẫu đang nói.

# Validate
assert topic.strip(), '❌ Nhập chủ đề video!'
assert groq_api_key.strip(), '❌ Nhập Groq API key!'
if not bridge_url.strip():
    assert pexels_api_key.strip(), '❌ Nhập Pexels API key hoặc Bridge URL!'

slug = re.sub(r'[^a-z0-9]+', '_', topic.lower()).strip('_')
print(f'✅ Cấu hình OK! Project: {slug}')
print(f'   Ảnh: {"TurboFlow AI" if bridge_url.strip() else "Pexels Stock"}')
print(f'   Giọng mẫu: {file_giong_mau}')
print(f'   → Chạy Cell 3 để tạo video')

In [ ]:
# @title 🚀 CELL 3: TẠO VIDEO (chạy 1 lần, ~5-10 phút)
import gc, base64
from IPython.display import Audio, display, HTML
from pydub import AudioSegment

P = Path(f'/content/{slug}')
P.mkdir(parents=True, exist_ok=True)
(P / 'images').mkdir(exist_ok=True)

print(f"{'='*60}")
print(f'🎬 SHORT MAKER: {topic}')
print(f"{'='*60}")

# ═══════════════════════════════════════════════════════════
# STEP 1: GENERATE SCRIPT (Groq)
# ═══════════════════════════════════════════════════════════
print(f'\n📖 [1/7] Sinh kịch bản (Groq AI)...')

PROMPT = '''You are a top-tier viral YouTube Shorts scriptwriter and English language educator. Write an engaging English learning script about: "{topic}"
Rules:
- 120-140 words, flowing narrative, NOT bullet points.
- Paragraph 1: Hook. Paragraphs 2-4: Body. Last: CTA.
- COMMAS = no pause. PERIODS/QUESTIONS = brief pause. BLANK LINES = dramatic pause.
- NEVER use ellipsis (...).
Extract 10-12 visual keywords from YOUR script.
Output valid JSON only:
{{"script": "...", "visual_keywords": [{{"keyword": "...", "search_query": "..."}}]}}'''

resp = requests.post(
    'https://api.groq.com/openai/v1/chat/completions',
    headers={'Authorization': f'Bearer {groq_api_key}', 'Content-Type': 'application/json'},
    json={'model': 'llama-3.3-70b-versatile',
          'messages': [{'role': 'system', 'content': 'Return valid JSON only.'},
                       {'role': 'user', 'content': PROMPT.format(topic=topic)}],
          'response_format': {'type': 'json_object'}, 'temperature': 0.8, 'max_tokens': 2000},
    timeout=30)
resp.raise_for_status()
raw = resp.json()['choices'][0]['message']['content'].strip()
raw = re.sub(r'^```json\s*', '', raw); raw = re.sub(r'\s*```$', '', raw)
script_data = json.loads(raw)
script_text = script_data['script']
keywords = script_data.get('visual_keywords', [])
(P / 'script.txt').write_text(script_text, encoding='utf-8')
(P / 'keywords.json').write_text(json.dumps(script_data, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'   ✅ {len(script_text.split())} từ, {len(keywords)} visual keywords')

# ═══════════════════════════════════════════════════════════
# STEP 2: GENERATE VOICE (Qwen3-TTS Voice Clone)
# ═══════════════════════════════════════════════════════════
print(f'\n🎤 [2/7] Tạo giọng đọc (Voice Clone từ {file_giong_mau})...')

assert os.path.exists(file_giong_mau), f'❌ Chưa thấy file giọng mẫu: {file_giong_mau}. Upload lên Colab trước!'

tts_model = Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-1.7B-Base', torch_dtype=torch.float16,
    device_map='cuda:0', attn_implementation='sdpa')

print('   ⚡️ Đang học giọng mẫu...')
clone_prompt = tts_model.create_voice_clone_prompt(
    ref_audio=file_giong_mau, ref_text=loi_thoai_giong_mau.strip(), x_vector_only_mode=False)

nghi_ngan = AudioSegment.silent(duration=300)  # 0.3s giữa các câu
nghi_dai = AudioSegment.silent(duration=500)   # 0.5s giữa các đoạn
final_audio = AudioSegment.silent(duration=300) # 0.3s mở đầu

raw_text = script_text.replace('\r', '')
paragraphs = re.split(r'\n\s*\n', raw_text)
total_sentences = sum(len(re.split(r'(?<=[.?!;:\n])\s+', p.strip())) for p in paragraphs if p.strip())
line_count = 0

print(f'   🚀 Thu âm {total_sentences} câu...')
for p_text in paragraphs:
    p_text = p_text.strip()
    if not p_text: continue
    sentences = re.split(r'(?<=[.?!;:\n])\s+', p_text)
    for s in sentences:
        s = s.strip()
        if not s: continue
        line_count += 1
        print(f'   🎙️ [{line_count}/{total_sentences}]: {s[:60]}...')
        with torch.inference_mode():
            w, sr = tts_model.generate_voice_clone(text=s, voice_clone_prompt=clone_prompt)
        soundfile.write('/content/temp_line.wav', w[0], sr)
        final_audio += AudioSegment.from_wav('/content/temp_line.wav') + nghi_ngan
        if os.path.exists('/content/temp_line.wav'): os.remove('/content/temp_line.wav')
    final_audio += nghi_dai

mp3_path = str(P / 'audio.mp3')
final_audio.export(mp3_path, format='mp3')
del tts_model, clone_prompt; gc.collect(); torch.cuda.empty_cache()
print(f'   ✅ audio.mp3 đã tạo')
display(Audio(mp3_path, autoplay=False))

# ═══════════════════════════════════════════════════════════
# STEP 3: WORD-LEVEL SUBTITLES (Whisper)
# ═══════════════════════════════════════════════════════════
print(f'\n📝 [3/7] Tạo phụ đề từng từ (Whisper)...')
w_model = whisper.load_model('base.en')
result = w_model.transcribe(mp3_path, word_timestamps=True, language='en')
del w_model; gc.collect(); torch.cuda.empty_cache()

# SRT file
srt_lines = []
for i, seg in enumerate(result['segments'], 1):
    s, e = seg['start'], seg['end']
    srt_lines.append(f"{i}\n{int(s//3600):02d}:{int(s%3600//60):02d}:{int(s%60):02d},{int(s%1*1000):03d} --> {int(e//3600):02d}:{int(e%3600//60):02d}:{int(e%60):02d},{int(e%1*1000):03d}\n{seg['text'].strip()}\n")
(P / 'subtitle.srt').write_text('\n'.join(srt_lines), encoding='utf-8')

# Word timestamps JSON
word_ts = [{'word': w['word'].strip(), 'start': round(w['start'], 3), 'end': round(w['end'], 3)}
           for seg in result['segments'] for w in seg.get('words', [])]
(P / 'word_timestamps.json').write_text(json.dumps(word_ts, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'   ✅ subtitle.srt ({len(result["segments"])} câu) + word_timestamps.json ({len(word_ts)} từ)')

# ═══════════════════════════════════════════════════════════
# STEP 4: GET IMAGES (TurboFlow AI or Pexels Stock)
# ═══════════════════════════════════════════════════════════
print(f'\n🖼️ [4/7] Tải ảnh...')
images_dir = P / 'images'

if bridge_url.strip():
    # === TurboFlow AI Images via Bridge ===
    burl = bridge_url.strip().rstrip('/')
    print(f'   🎨 Kết nối TurboFlow: {burl}')
    health = requests.get(f'{burl}/health', timeout=5)
    assert health.ok, f'❌ Không kết nối được bridge! Check bridge_local.py'
    print(f'   ✅ Bridge connected!')

    # Send prompts to bridge
    prompts = [kw['search_query'] for kw in keywords]
    job_resp = requests.post(f'{burl}/enqueue', json={
        'prompts': prompts,
        'expected_count': len(prompts),
        'settings': {'naming': 'prefix', 'namingPrefix': slug, 'namingSeparator': '-'}
    }, timeout=10)
    job_data = job_resp.json()
    job_id = job_data['job_id']
    print(f'   📤 Đã gửi {len(prompts)} prompts → TurboFlow (job: {job_id})')
    print(f'   ⏳ Đang chờ TurboFlow gen ảnh... (mở Edge + extension + Google Flow)')

    # Poll for images
    for attempt in range(120):  # 4 min timeout
        time.sleep(2)
        imgs_resp = requests.get(f'{burl}/images', params={'job_id': job_id}, timeout=10)
        items = imgs_resp.json().get('items', [])
        print(f'\r   ⏳ Đã nhận {len(items)}/{len(prompts)} ảnh...', end='', flush=True)
        if len(items) >= len(prompts):
            break
    print()

    # Download images to Colab
    for i, item in enumerate(items):
        kw_slug = re.sub(r'[^a-z0-9]+', '_', keywords[i]['keyword'].lower()).strip('_') if i < len(keywords) else f'img_{i}'
        dest = images_dir / f'{kw_slug}.jpg'
        img_data = requests.get(f'{burl}/download', params={'name': item['name']}, timeout=30)
        dest.write_bytes(img_data.content)
    print(f'   ✅ Đã tải {len(items)} ảnh AI từ TurboFlow')

else:
    # === Pexels Stock Images ===
    print(f'   📷 Tải ảnh stock từ Pexels...')
    for kw in keywords:
        kw_slug = re.sub(r'[^a-z0-9]+', '_', kw['keyword'].lower()).strip('_')
        dest = images_dir / f'{kw_slug}.jpg'
        if dest.exists(): continue
        for query in [kw['search_query'], kw['keyword']]:
            try:
                r = requests.get('https://api.pexels.com/v1/search',
                    headers={'Authorization': pexels_api_key},
                    params={'query': query, 'orientation': 'portrait', 'per_page': 3, 'size': 'large'}, timeout=10)
                photos = r.json().get('photos', [])
                if photos:
                    img_url = photos[0]['src'].get('portrait') or photos[0]['src']['large2x']
                    img_data = requests.get(img_url, timeout=30)
                    dest.write_bytes(img_data.content)
                    print(f'   ✅ {kw_slug}.jpg')
                    break
            except: pass
    print(f'   ✅ Ảnh Pexels đã tải')

# ═══════════════════════════════════════════════════════════
# STEP 5: BUILD VISUAL TIMELINE
# ═══════════════════════════════════════════════════════════
print(f'\n🎞️ [5/7] Xây dựng timeline...')

subtitles = []
subs_raw = pysrt.open(str(P / 'subtitle.srt'), encoding='utf-8')
for sub in subs_raw:
    subtitles.append({'index': sub.index, 'text': sub.text.replace('\n', ' ').strip(),
                      'start': sub.start.ordinal / 1000.0, 'end': sub.end.ordinal / 1000.0})

# Get audio duration
dur_result = subprocess.run(['ffmpeg', '-i', mp3_path], capture_output=True, text=True, errors='ignore')
dur_match = re.search(r'Duration:\s*(\d+):(\d+):(\d+\.\d+)', dur_result.stderr)
audio_duration = int(dur_match.group(1))*3600 + int(dur_match.group(2))*60 + float(dur_match.group(3)) if dur_match else subtitles[-1]['end']

# Align words to subtitles
def normalize_token(t): return re.sub(r'[^a-z0-9]+', '', t.lower())
def tokenize_text(t): return [x for x in (normalize_token(w) for w in re.split(r'\s+', t)) if x]

words_norm = [normalize_token(w['word']) for w in word_ts]
cursor = 0
for sub in subtitles:
    scan_start = cursor
    tokens = tokenize_text(sub['text'])
    if not tokens: sub['words'] = []; continue
    match_indices = []
    for token in tokens:
        while cursor < len(words_norm) and words_norm[cursor] != token: cursor += 1
        if cursor >= len(words_norm): break
        match_indices.append(cursor); cursor += 1
    ratio = len(match_indices) / max(len(tokens), 1)
    if match_indices and ratio >= 0.7:
        sub['start'] = max(0.0, word_ts[match_indices[0]]['start'])
        sub['end'] = word_ts[match_indices[-1]]['end']
        sub['words'] = word_ts[match_indices[0]:match_indices[-1]+1]
        continue
    cursor = match_indices[-1]+1 if match_indices else scan_start
    sub['words'] = [w for w in word_ts if sub['start'] <= w['start'] <= sub['end']]

# Build image timeline
image_paths = []
for kw in keywords:
    kw_slug = re.sub(r'[^a-z0-9]+', '_', kw['keyword'].lower()).strip('_')
    img = images_dir / f'{kw_slug}.jpg'
    if img.exists():
        image_paths.append(f'file:///{str(img.resolve())}')

segments = []
if image_paths:
    for idx, sub in enumerate(subtitles):
        segments.append({'type': 'image', 'path': image_paths[idx % len(image_paths)],
                         'start': sub['start'], 'end': sub['end'], 'slot': 1 if idx%2==0 else 2})
    if segments:
        segments[0]['start'] = 0.0
        for i in range(len(segments)-1): segments[i]['end'] = segments[i+1]['start']
        segments[-1]['end'] = audio_duration

keywords_set = {kw['keyword'].lower() for kw in keywords}
formatted_subs = []
for sub in subtitles:
    hl = None
    for kw in keywords_set:
        if re.search(r'\b' + re.escape(kw) + r'\b', sub['text'], re.IGNORECASE): hl = kw; break
    formatted_subs.append({'start': sub['start'], 'end': sub['end'], 'text': sub['text'],
                           'highlighted': hl, 'words': sub.get('words', [])})

video_data = {'title': slug.replace('_', ' ').title(), 'segments': segments, 'subtitles': formatted_subs}
print(f'   ✅ {len(segments)} segments, {len(formatted_subs)} subtitles')

# ═══════════════════════════════════════════════════════════
# STEP 6: RENDER VIDEO (Playwright + FFmpeg)
# ═══════════════════════════════════════════════════════════
print(f'\n🎬 [6/7] Render video (Playwright)...')

template_path = P / 'template_v2.html'

TEMPLATE_CONTENT = '''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=1080, height=1920, initial-scale=1.0">
    <title>Viral English Shorts Template</title>
    <style>
        * { box-sizing: border-box; margin: 0; padding: 0; }
        body, html { width: 1080px; height: 1920px; overflow: hidden; background-color: #0b0914; font-family: 'Montserrat', Arial, sans-serif; }
        #bg-gradient { position: absolute; top: 0; left: 0; width: 1080px; height: 1920px; background: radial-gradient(circle at 50% 50%, #1e1b4b 0%, #0b0914 100%); z-index: 1; }
        .glow { position: absolute; width: 800px; height: 800px; background: radial-gradient(circle, rgba(99, 102, 241, 0.15) 0%, transparent 70%); top: 30%; left: 10%; z-index: 2; pointer-events: none; }
        #header-container { position: absolute; top: 80px; left: 50%; transform: translateX(-50%); background: linear-gradient(135deg, rgba(255,255,255,0.1), rgba(255,255,255,0.03)); backdrop-filter: blur(10px); padding: 15px 40px; border-radius: 50px; z-index: 20; border: 1px solid rgba(255, 255, 255, 0.15); text-align: center; box-shadow: 0 10px 30px rgba(0,0,0,0.3); white-space: nowrap; }
        #header-text { font-size: 32px; font-weight: 900; color: #fff; letter-spacing: 4px; text-transform: uppercase; background: linear-gradient(90deg, #00f0ff, #ff007f); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
        #media-wrapper { position: absolute; top: 180px; left: 50%; transform: translateX(-50%); width: 860px; height: 860px; z-index: 10; display: flex; justify-content: center; align-items: center; }
        #media-container { width: 800px; height: 800px; border-radius: 40px; overflow: hidden; box-shadow: 0 35px 70px rgba(0,0,0,0.7), 0 0 45px rgba(99, 102, 241, 0.25); border: 6px solid rgba(255,255,255,0.15); background: #121026; position: relative; }
        .media-img { position: absolute; top: 0; left: 0; width: 100%; height: 100%; object-fit: cover; transform-origin: center center; opacity: 0; }
        #text-pop-container { position: absolute; top: 0; left: 0; width: 100%; height: 100%; display: flex; justify-content: center; align-items: center; background: linear-gradient(135deg, #3b0764 0%, #121026 100%); opacity: 0; z-index: 5; padding: 40px; text-align: center; }
        #text-pop-word { font-size: 96px; font-weight: 900; color: #ffff00; text-transform: uppercase; text-shadow: 0 0 40px rgba(255, 255, 0, 0.8), 0 5px 20px rgba(0,0,0,0.8); letter-spacing: 4px; transform-origin: center center; }
        #subtitle-container { position: absolute; top: 1080px; left: 50%; transform: translateX(-50%); width: 960px; height: 680px; display: flex; justify-content: center; align-items: center; text-align: center; z-index: 20; padding: 20px; }
        .subtitle-text { font-size: 78px; font-weight: 900; line-height: 1.4; color: #ffffff; text-transform: uppercase; display: flex; flex-wrap: wrap; justify-content: center; gap: 20px 30px; text-shadow: 0 4px 10px rgba(0,0,0,0.9); }
        .word { display: inline-block; transform: scale(0.9); opacity: 0.4; }
        .word.active { color: #ffff00; transform: scale(1.25); opacity: 1; text-shadow: 0 0 30px rgba(255, 255, 0, 0.6), 0 4px 15px rgba(0,0,0,0.9); }
        .word.keyword-match { color: #00f0ff; opacity: 0.85; }
        .word.active.keyword-match { color: #00ff66; transform: scale(1.3); text-shadow: 0 0 35px rgba(0, 255, 102, 0.7), 0 4px 15px rgba(0,0,0,0.9); }
    </style>
</head>
<body>
    <div id="bg-gradient"></div>
    <div class="glow"></div>
    <div id="header-container"><div id="header-text">English Pro</div></div>
    <div id="media-wrapper">
        <div id="media-container">
            <img id="media-image-1" class="media-img" src="" alt="">
            <img id="media-image-2" class="media-img" src="" alt="">
            <div id="text-pop-container"><div id="text-pop-word">WORD</div></div>
        </div>
    </div>
    <div id="subtitle-container"><div class="subtitle-text" id="subtitle-display"></div></div>
    <script>
        window.videoData = window.videoData || { title: "Learn English", segments: [], subtitles: [] };
        const bgGradient = document.getElementById("bg-gradient");
        const img1 = document.getElementById("media-image-1");
        const img2 = document.getElementById("media-image-2");
        const textPopContainer = document.getElementById("text-pop-container");
        const textPopWord = document.getElementById("text-pop-word");
        const subtitleDisplay = document.getElementById("subtitle-display");
        const headerText = document.getElementById("header-text");
        const mediaWrapper = document.getElementById("media-wrapper");
        let img1Src = "", img2Src = "";
        const imageCache = {};
        function easeInOutSine(t) { return -(Math.cos(Math.PI * t) - 1) / 2; }
        async function preloadImage(src) { if (imageCache[src]) return; const img = new Image(); img.crossOrigin = "anonymous"; await new Promise(r => { img.onload = r; img.onerror = r; img.src = src; }); if (img.decode) await img.decode().catch(() => null); imageCache[src] = img; }
        function setImgSrc(which, newSrc) { const elem = which === "img1" ? img1 : img2; const currentSrc = which === "img1" ? img1Src : img2Src; if (currentSrc === newSrc) return; elem.src = newSrc; if (which === "img1") img1Src = newSrc; else img2Src = newSrc; }
        function applySegmentStyles(elem, type, seg, timeSeconds, targetOpacity) {
            elem.style.opacity = targetOpacity; if (targetOpacity <= 0) return;
            const duration = seg.end - seg.start, elapsed = timeSeconds - seg.start;
            if (type === "image") { const t = timeSeconds * 0.35; elem.style.transform = `scale(${1.03 + 0.015 * Math.sin(t)}) rotate(${0.25 * Math.sin(t * 0.8 + 1.2)}deg)`; }
            else if (type === "text_pop") { let scale = 1.0; const popTime = 0.22; if (elapsed < popTime) { scale = 0.4 + 0.9 * Math.sin((elapsed / popTime) * Math.PI / 2); } else { scale = 1.3 - 0.1 * ((elapsed - popTime) / (duration - popTime || 1)); } textPopWord.innerText = seg.text; textPopWord.style.transform = `scale(${scale})`; }
        }
        window.renderFrame = function(timeSeconds) {
            if (headerText.innerText !== window.videoData.title) headerText.innerText = window.videoData.title;
            bgGradient.style.background = `radial-gradient(circle at 50% 50%, #1e1b4b 0%, #0b0914 100%)`;
            const floatOffset = Math.sin(timeSeconds * 1.2) * 6;
            mediaWrapper.style.transform = `translateX(-50%) translateY(${floatOffset}px)`;
            let activeIdx = -1;
            for (let i = 0; i < window.videoData.segments.length; i++) { const seg = window.videoData.segments[i]; if (timeSeconds >= seg.start && timeSeconds <= seg.end) { activeIdx = i; break; } }
            if (activeIdx !== -1) {
                const activeSeg = window.videoData.segments[activeIdx];
                const segDur = Math.max(activeSeg.end - activeSeg.start, 0.001);
                const crossDur = Math.min(0.35, segDur * 0.5);
                const tInSeg = timeSeconds - activeSeg.start;
                if (activeIdx > 0 && crossDur > 0 && tInSeg < crossDur) {
                    const prevSeg = window.videoData.segments[activeIdx - 1];
                    const progress = easeInOutSine(tInSeg / crossDur);
                    if (prevSeg.type === "image") { const pSlot = prevSeg.slot === 1 ? "img1" : "img2"; setImgSrc(pSlot, prevSeg.path); applySegmentStyles(prevSeg.slot === 1 ? img1 : img2, "image", prevSeg, timeSeconds, 1 - progress); }
                    else if (prevSeg.type === "text_pop") { applySegmentStyles(textPopContainer, "text_pop", prevSeg, timeSeconds, 1 - progress); }
                    if (activeSeg.type === "image") { const aSlot = activeSeg.slot === 1 ? "img1" : "img2"; const aElem = activeSeg.slot === 1 ? img1 : img2; const oElem = activeSeg.slot === 1 ? img2 : img1; setImgSrc(aSlot, activeSeg.path); applySegmentStyles(aElem, "image", activeSeg, timeSeconds, progress); if (prevSeg.type !== "image") { oElem.style.opacity = 0; textPopContainer.style.opacity = 0; } }
                    else if (activeSeg.type === "text_pop") { applySegmentStyles(textPopContainer, "text_pop", activeSeg, timeSeconds, progress); if (prevSeg.type !== "image") { img1.style.opacity = 0; img2.style.opacity = 0; } }
                } else {
                    if (activeSeg.type === "image") { const aSlot = activeSeg.slot === 1 ? "img1" : "img2"; const aElem = activeSeg.slot === 1 ? img1 : img2; const oElem = activeSeg.slot === 1 ? img2 : img1; setImgSrc(aSlot, activeSeg.path); applySegmentStyles(aElem, "image", activeSeg, timeSeconds, 1.0); oElem.style.opacity = 0; textPopContainer.style.opacity = 0; }
                    else if (activeSeg.type === "text_pop") { img1.style.opacity = 0; img2.style.opacity = 0; applySegmentStyles(textPopContainer, "text_pop", activeSeg, timeSeconds, 1.0); }
                }
            } else { img1.style.opacity = 0; img2.style.opacity = 0; textPopContainer.style.opacity = 0; }
            let activeSub = null;
            for (let i = 0; i < window.videoData.subtitles.length; i++) { const sub = window.videoData.subtitles[i]; if (timeSeconds >= sub.start && timeSeconds <= sub.end) { activeSub = sub; break; } }
            if (activeSub) {
                let html = "";
                if (activeSub.words && activeSub.words.length > 0) {
                    let awi = -1;
                    for (let i = 0; i < activeSub.words.length; i++) { if (timeSeconds >= activeSub.words[i].start && timeSeconds <= activeSub.words[i].end) { awi = i; break; } }
                    if (awi === -1) { for (let i = 0; i < activeSub.words.length; i++) { if (timeSeconds >= activeSub.words[i].start) awi = i; } }
                    activeSub.words.forEach((w, idx) => { let cw = w.word.replace(/[.,\\/#!$%\\^&*;:{}=\\-_`~()?]/g,"").toLowerCase(); const isKw = activeSub.highlighted && cw.includes(activeSub.highlighted.toLowerCase()); let cn = "word"; if (idx === awi) cn += " active"; if (isKw) cn += " keyword-match"; html += `<span class="${cn}">${w.word}</span> `; });
                } else {
                    const words = activeSub.text.split(/\\s+/); const dur = activeSub.end - activeSub.start; const elapsed = timeSeconds - activeSub.start; const totalChars = words.reduce((s, w) => s + w.length, 0) || 1; let cumTime = 0; let awi = -1;
                    const wt = words.map(w => { const wd = dur * (w.length / totalChars); const st = cumTime; cumTime += wd; return { start: st, end: cumTime }; });
                    for (let i = 0; i < wt.length; i++) { if (elapsed >= wt[i].start && elapsed <= wt[i].end) { awi = i; break; } }
                    words.forEach((w, idx) => { let cw = w.replace(/[.,\\/#!$%\\^&*;:{}=\\-_`~()?]/g,"").toLowerCase(); const isKw = activeSub.highlighted && cw.includes(activeSub.highlighted.toLowerCase()); let cn = "word"; if (idx === awi) cn += " active"; if (isKw) cn += " keyword-match"; html += `<span class="${cn}">${w}</span> `; });
                }
                if (subtitleDisplay.innerHTML !== html) subtitleDisplay.innerHTML = html;
            } else { subtitleDisplay.innerHTML = ""; }
        };
        window.isReady = true;
    </script>
</body>
</html>'''

template_path.write_text(TEMPLATE_CONTENT, encoding='utf-8')

fps = 30
total_frames = int(audio_duration * fps) + 1
frames_dir = P / 'frames'
if frames_dir.exists(): shutil.rmtree(str(frames_dir))
frames_dir.mkdir()

with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    page = browser.new_page(viewport={'width': 1080, 'height': 1920})
    page.add_init_script(f'window.videoData = {json.dumps(video_data)};')
    page.goto(f'file:///{str(template_path.resolve())}')
    page.wait_for_function('window.isReady === true')

    # Preload images
    unique_imgs = set(s['path'] for s in segments if s.get('type') == 'image' and s.get('path'))
    if unique_imgs:
        print(f'   🖼️ Pre-loading {len(unique_imgs)} images...')
        preload_js = '; '.join([f"await window.preloadImage('{p}')" for p in unique_imgs])
        page.evaluate(f'(async () => {{ {preload_js} }})()')

    print(f'   Rendering {total_frames} frames @ {fps}fps...')
    t0 = time.time()
    for fi in range(total_frames):
        t = fi / fps
        page.evaluate(f'window.renderFrame({t})')
        page.screenshot(path=str(frames_dir / f'frame_{fi:05d}.png'), type='png')
        if (fi + 1) % 60 == 0:
            elapsed = time.time() - t0
            speed = (fi + 1) / elapsed if elapsed > 0 else 0
            print(f'   📸 {fi+1}/{total_frames} frames ({speed:.1f} fps)...')
    browser.close()
print(f'   ✅ {total_frames} frames rendered')

# ═══════════════════════════════════════════════════════════
# STEP 7: ENCODE VIDEO (FFmpeg)
# ═══════════════════════════════════════════════════════════
print(f'\n🎬 [7/7] Encoding video (FFmpeg)...')
output_path = P / 'output.mp4'
cmd = ['ffmpeg', '-y', '-framerate', str(fps), '-i', f'{frames_dir}/frame_%05d.png',
       '-i', mp3_path, '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
       '-c:a', 'aac', '-b:a', '192k', '-pix_fmt', 'yuv420p', '-movflags', '+faststart',
       str(output_path)]
enc_result = subprocess.run(cmd, capture_output=True, text=True, errors='ignore')
assert enc_result.returncode == 0, f'FFmpeg error: {enc_result.stderr[-500:]}'
shutil.rmtree(str(frames_dir), ignore_errors=True)

size_mb = output_path.stat().st_size / (1024*1024)
print(f'\n{"="*60}')
print(f'🎉 VIDEO HOÀN TẤT! ({size_mb:.1f} MB)')
print(f'   📁 {output_path}')
print(f'{"="*60}')
print(f'→ Chạy Cell 4 để tải video về máy')

In [ ]:
# @title 📥 CELL 4: TẢI VIDEO VỀ MÁY
from google.colab import files
files.download(str(output_path))
print('✅ Đang tải video...')